<a href="https://colab.research.google.com/github/ashishsantikari/lab-sql-query-from-table-names/blob/main/lab-sql-query-from-table-names.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from google.colab import userdata
# from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv())


# OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
OPENAI_API_KEY  = userdata.get('OPENAI_API_KEY')

In [15]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [16]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ["students", "marks", "fees", "admission"],
        'definition': [
            "Information about the students currently enrolled in our school",
            "A table to persist marks obtained by students",
            "Table showing fee details and history of fees paid by students",
            "Table storing information about the student admission"
        ]
        }
df = pd.DataFrame(data)
print(df)

       table                                         definition
0   students  Information about the students currently enrol...
1      marks      A table to persist marks obtained by students
2       fees  Table showing fee details and history of fees ...
3  admission  Table storing information about the student ad...


In [18]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [19]:
print(text_tables)

students: Information about the students currently enrolled in our school
marks: A table to persist marks obtained by students
fees: Table showing fee details and history of fees paid by students
admission: Table storing information about the student admission


In [21]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Question:
{question}
"""


In [22]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="what is the average revenue generated by fees?")

In [23]:
print(return_OAI(pqt1))

{
    "tables": ["fees"]
}


In [27]:
pqt3 = prompt_question_tables.format(tables=text_tables,question="what is the total number of students studying in school?")

In [29]:
print(return_OAI(pqt3))

```json
{
    "tables": ["students"]
}
```


In [33]:
pqt4 = prompt_question_tables.format(tables=text_tables,question="what is the total amount of fees collected in last quarter?")
print(return_OAI(pqt4))

```json
{
    "tables": ["fees"]
}
```


In [34]:
pqt5 = prompt_question_tables.format(tables=text_tables,question="what is the average age of students by class?")
print(return_OAI(pqt5))

```json
{
    "tables": ["students"]
}
```


In [35]:
pqt6 = prompt_question_tables.format(tables=text_tables,question="what is the ratio of boys to girls in school?")
print(return_OAI(pqt6))

```json
{
    "tables": ["students"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

From the description, the model is able to detect the right set of findings and is able to provide correct answers.
